# 埋め込み（Embeddings）を使用したセマンティック マッチング

**学習目標**
  1. Google Gen AI SDK で Gemini 埋め込みを使用する方法を学ぶ
  2. さまざまな類似度指標を計算する方法を学ぶ
  3. 説明文から生成された埋め込みを使用して一致する製品を特定する方法を学ぶ
  4. セマンティック マッチングのパフォーマンスを評価する方法を学ぶ
  
セマンティック マッチングとは、エンティティのペア $(x_1, x_2)$ が適切な一致（マッチ）であるかどうかを分類する問題です。これは非常に柔軟な分類設定であり、一般的な情報検索（最初のエンティティがテキスト クエリで、2 つ目のエンティティが段落である場合など）、エンティティ解決、データベース レコードのあいまい一致などを幅広く包含します。このノートブックでは、小売製品のテキスト説明のマッチングに焦点を当てます。具体的には以下のとおりです。
  
  
**ユースケースの説明:** あるオンライン小売企業は、自社の在庫商品の価格と競合他社が提供する価格を比較するためにウェブを調査しています。同社の最優先事項は、2 つの製品ウェブページの情報を比較し、ページ上の異なる製品説明が同一の製品に対応しているかどうか（これを「マッチ（一致）」と呼びます）を示す分類を出力するモデルを実装することです。ここでは、ライプチヒ大学によって作成された [エンティティ解決ベンチマーク](https://dbs.uni-leipzig.de/research/projects/object_matching/benchmark_datasets_for_entity_resolution) 内の [Amazon-Google Products データセット](https://dbs.uni-leipzig.de/file/Amazon-GoogleProducts.zip) を使用します。

**モデルの説明:** このノートブックでは、Agent Platform で Gemini Embeddings API を使用して製品説明をマッチングする方法を説明します。[Amazon-GoogleProducts データセット](https://dbs.uni-leipzig.de/file/Amazon-GoogleProducts.zip) には、Amazon または Google のウェブサイトからスクレイピングされた製品情報が含まれています。製品のタイトル、説明、価格、メーカーが含まれますが、これら 2 つのウェブサイトでは表現が異なっています。このノートブックでは、製品のタイトルと説明を使用したモデルの構築に焦点を当てます。これらのテキスト フィールドを埋め込みに変換し、類似度指標で評価することで、掲載内容の表現がまったく異なる場合でも、製品ペアを高い信頼性でマッチングする方法を学びます。

## セットアップ

In [ ]:
import os
import random
import numpy as np
import pandas as pd
from google import genai
from google.genai import types

pd.options.display.max_colwidth = 1000

In [ ]:
REGION = "us-central1"
PROJECT = !(gcloud config get-value core/project)
PROJECT = PROJECT[0]

# これらは変更しないでください
os.environ["PROJECT"] = PROJECT
os.environ["REGION"] = REGION

### テキスト埋め込み（Text Embeddings）API
推奨されるモデルは `gemini-embedding-001` であり、英語、多言語、コードの各タスクにわたって最大 3072 次元の出力で最先端のパフォーマンスを提供します。より特殊なユースケースとして、`text-embedding-005` は英語およびコードのタスクに特化しており、`text-multilingual-embedding-002` は多言語アプリケーションに焦点を当てています。3 つのモデルはすべて、最大シーケンス長 2048 トークンをサポートしています。
詳細情報:
https://docs.cloud.google.com/gemini-enterprise-agent-platform/reference/models/text-embeddings-api

In [ ]:
EMBEDDING_MODEL = "gemini-embedding-001" 
#EMBEDDING_MODEL = "text-embedding-005"
#EMBEDDING_MODEL = "text-multilingual-embedding-002"

### テキスト埋め込みのタスク タイプ（Task Types）
次の表は、task_type パラメータの値とそのユースケースを説明しています。

| `Task type` | 説明 |
|---|---|
| **`RETRIEVAL_QUERY`** | 指定されたテキストが検索または情報取得設定におけるクエリであることを指定します。ドキュメント側には `RETRIEVAL_DOCUMENT` を使用します。 |
| **`RETRIEVAL_DOCUMENT`** | 指定されたテキストが検索または情報取得設定におけるドキュメントであることを指定します。 |
| **`SEMANTIC_SIMILARITY`** | 指定されたテキストがテキスト意味的類似性（STS: Semantic Textual Similarity）に使用されることを指定します。 |
| **`CLASSIFICATION`** | 埋め込みが分類に使用されることを指定します。 |
| **`CLUSTERING`** | 埋め込みがクラスタリングに使用されることを指定します。 |
| **`QUESTION_ANSWERING`** | クエリの埋め込みが質問への回答に使用されることを指定します。ドキュメント側には `RETRIEVAL_DOCUMENT` を使用します。 |
| **`FACT_VERIFICATION`** | クエリの埋め込みが事実確認に使用されることを指定します。ドキュメント側には `RETRIEVAL_DOCUMENT` を使用します。 |
| **`CODE_RETRIEVAL_QUERY`** | クエリの埋め込みが Java および Python のコード検索に使用されることを指定します。ドキュメント側には `RETRIEVAL_DOCUMENT` を使用します。 |

---

### 🔍 検索タスク（Retrieval Tasks）

* **クエリ（Query）**: 入力テキストが検索クエリであることを示すには `task_type=RETRIEVAL_QUERY` を使用します。
* **コーパス（Corpus）**: 入力テキストが検索対象のドキュメント コレクションの一部であることを示すには `task_type=RETRIEVAL_DOCUMENT` を使用します。

### 類似度タスク（Similarity Tasks）

* **セマンティック類似度（Semantic similarity）**: 両方の入力テキストに `task_type=SEMANTIC_SIMILARITY` を使用して、全体的な意味の類似度を評価します。

In [ ]:
#TODO: 埋め込みタスク タイプを選択
EMBD_TASK_TYPE="..." 

### クライアントの初期化:
パラメータ vertexai=True を指定し、Google Cloud プロジェクト ID とリージョンを提供して Google Gen AI クライアントを使用します

In [ ]:
client = genai.Client(
    vertexai=True,
    project=PROJECT,
    location="global"
)

## 埋め込みの生成
### テキスト文字列から埋め込みを作成するユーティリティ メソッドの定義:

In [ ]:
def embed_text(text_to_embed: str | list[str], output_dimensionality: int = None) -> list[list[float]]:
    """
    Generates embeddings for the provided text.
    """
    try:
        response = client.models.embed_content(
            model=EMBEDDING_MODEL,
            contents=text_to_embed,
            config=types.EmbedContentConfig(
                task_type=EMBD_TASK_TYPE,
                output_dimensionality=output_dimensionality 
            )
        )
        
        return [np.array(embed.values) for embed in response.embeddings]

    except Exception as e:
        print(f"Failed to generate embeddings: {e}")

### embed_text メソッドの使用例:

In [ ]:
text_to_embed = "emc retrospect 7.5 disk to diskcromwindows"

test_embedding = embed_text(text_to_embed)
print(f"Generated a vector with {len(test_embedding[0])} dimensions.")
#TODO: 最初のベクトルの先頭 5 つの浮動小数点数を表示するコードを実装
print(f"View the first 5 floats: ...") # 
#TODO: 最初のベクトルを取得するコードを実装
input_embd_vector=test_embedding[0]

### 埋め込みベクトルの正規化（L2 ノルム）

`np.linalg.norm()` を使用して L2 ノルム（ユークリッド長）を計算してみましょう。ご覧のように、結果として得られるベクトルの大きさは約 `1.0` であり、埋め込みが単位正規化されていることが確認できます。正規化された埋め込みは、2 つのベクトル間のコサイン類似度を単純な内積で直接計算できるため、ダウンストリーム タスクを簡素化します。

In [ ]:
magnitude = np.linalg.norm(input_embd_vector)
print(f"vector magnitude = {magnitude}")

### MRL 埋め込みの L2 正規化
**重要な正規化ルール**: API は、デフォルト サイズの出力ベクトルを自動的に L2 正規化します。ただし、output_dimensionality を使用してより小さいサイズ（128 や 768 など）を要求した場合、API は生の正規化されていない数値を返します。コサイン類似度の代わりに内積を使用したい場合は、コード内で切り捨てられたベクトルを手動で正規化する必要があります。

#### オプション 1. output_dimensionality パラメータを使用する:

In [ ]:
#TODO: サイズ 128 の埋め込みを生成するコードを実装:
embedding_mrl_128 = embed_text(...)
print(f"Generated a vector with {len(embedding_mrl_128[0])} dimensions.")
magnitude = np.linalg.norm(embedding_mrl_128)
print(f"vector magnitude = {magnitude}")

#### オプション 2. 単に切り捨てる:

In [ ]:
embedding_trunc_128 = input_embd_vector[:128]
print(f"Generated a vector with {len(embedding_trunc_128)} dimensions.")
magnitude = np.linalg.norm(embedding_trunc_128)
print(f"vector magnitude = {magnitude}")

### 埋め込みの L2 正規化

**ベクトル $L_2$ 正規化** は、非ゼロベクトルのユークリッド長（大きさ）の合計が正確に $1$ になるようにスケーリングします。このプロセスにより、任意のベクトルがまったく同じ方向を指す **単位ベクトル** に変換されます。

機械学習や埋め込みのワークフローにおいて、ベクトルを単位長に正規化することは非常に一般的な前処理ステップです。ベクトルが $L_2$ 正規化されている場合、それらの **コサイン類似度** の計算は、シンプルで計算効率の高い **内積（Dot Product）** に単純化されます。
$L_2$ 正規化の式は、ベクトル $v$ をそのユークリッド（$L_2$）ノルムで割ります。

$$\hat{v} = \frac{v}{\|v\|_2}$$

#### 1. $L_2$ ノルム（$\|v\|_2$）の計算
分母はユークリッド長であり、各要素の 2 乗の合計の平方根として計算されます。

$$\|v\|_2 = \sqrt{\sum_{i=1}^{n} v_i^2} = \sqrt{v_1^2 + v_2^2 + \dots + v_n^2}$$

#### 2. 要素ごとの除算
スカラー ノルムが求まったら、ベクトル内のすべての要素をその値で割ります。

$$\hat{v} = \left[ \frac{v_1}{\|v\|_2}, \frac{v_2}{\|v\|_2}, \dots, \frac{v_n}{\|v\|_2} \right]$$

#### ⚙️ L2 正規化の実装

NumPy を使用したこの関数の基本的な実装はシンプルです。

In [ ]:
#TODO: L2 正規化のコードを実装:
def l2_normalize(vec):
    """
    Normalizes a vector to unit length (L2 Norm = 1).
    """
    ...

#### L2 正規化の実装のテスト

In [ ]:
# テスト 1: 標準的な 3D ベクトル（[3, 4] のノルムは 5 なので正規化後は [0.6, 0.8]）
res1 = l2_normalize([3.0, 4.0])
assert np.allclose(res1, [0.6, 0.8]), f"Failed Standard: Expected [0.6, 0.8], got {res1}"

# テスト 2: 単位ベクトルのチェック（正常に正規化されたベクトルのノルムは正確に 1）
assert np.isclose(np.linalg.norm(res1), 1.0), "Failed Unit Length check"

# テスト 3: ゼロベクトルの安全性（ZeroDivisionError を発生させずにゼロを返す）
res2 = l2_normalize([0.0, 0.0])
assert np.allclose(res2, [0.0, 0.0]), f"Failed Zero Safety: Expected [0.0, 0.0], got {res2}"

# テスト 4: 既に正規化されたベクトル（変更されないこと）
res3 = l2_normalize([1.0, 0.0])
assert np.allclose(res3, [1.0, 0.0]), f"Failed Already Normalized: Expected [1.0, 0.0], got {res3}"

print("✅ All quick asserts passed successfully!")

L2 正規化後のベクトルの大きさを確認:

In [ ]:
embedding_l2_norm_mrl_128 = l2_normalize(embedding_trunc_128)
magnitude = np.linalg.norm(embedding_l2_norm_mrl_128)
print(f"vector magnitude = {magnitude}")

### 埋め込みの可視化:

In [ ]:
import plotly.express as px
import numpy as np

# Plot the vector as a sequence of bars
fig = px.bar(
    x=range(len(input_embd_vector)),
    y=input_embd_vector,
    # Crucial change: pass the values to 'color' to map them to a continuous scale
    color=input_embd_vector, 
    # Use a diverging color scale (Red for positive, Blue for negative)
    color_continuous_scale="RdBu_r", 
    labels={"x": f"Vector Dimension (0 - {len(input_embd_vector)})", "y": "Activation Value", "color": "Value"},
    title="Vector Visualization (Color-Coded Activation)",
    template="plotly_white"
)

# Style the bars to be sharp and distinct
fig.update_traces(marker_line_width=0)
fig.update_layout(
    bargap=0.1,
    coloraxis_cmid=0 
)

fig.show()

### 製品説明を含む最初のデータセットの読み込み
Google および Amazon のウェブサイトからスクレイピングされた [製品情報のデータセット](https://dbs.uni-leipzig.de/file/Amazon-GoogleProducts.zip) を使用します。このデータセットは、ライプチヒ大学による [セマンティック マッチングおよびエンティティ解決のベンチマーク](https://dbs.uni-leipzig.de/research/projects/object_matching/benchmark_datasets_for_entity_resolution) の一部です。このリポジトリに含まれる次の 3 つのテーブルで構成されています。

```python
../data/Amazon.csv.gz 
../data/GoogleProducts.csv.gz
../Amzon_GoogleProducts_perfectMapping.csv.gz

```

最初のテーブルには、製品のタイトル、説明、メーカーなど、Amazon に掲載されている製品情報が含まれています。

In [ ]:
amazon = pd.read_csv("../data/Amazon.csv.gz")
amazon.columns = [
    "idAmazon",
    "amazon_title",
    "amazon_description",
    "amazon_manufacturer",
    "amazon_price",
]
amazon = amazon.dropna(subset=['amazon_description'])
amazon.head()

### gemini-embedding-001 モデルを使用して結合されたタイトルと説明の埋め込みを作成

In [ ]:
import time
from tqdm import tqdm

BATCH_SIZE = 100 

# 1. タイトルと説明を結合（欠損値を適切に処理）
combined_text = amazon['amazon_title'].fillna('') + " " + amazon['amazon_description'].fillna('')

# リストに変換し、先ほど使用した切り捨てロジックを適用
MAX_CHARS = 4000
descriptions = [str(text)[:MAX_CHARS] for text in combined_text.tolist()]

all_embeddings = []

print(f"Starting to embed {len(descriptions)} documents in batches of {BATCH_SIZE}...")

# 3. チャンク単位でリストをループ処理
for i in tqdm(range(0, len(descriptions), BATCH_SIZE), desc="Embedding Batches"):
    # リストをスライスして現在のバッチを取得
    batch_texts = descriptions[i : i + BATCH_SIZE]
    
    # embed_text 関数を呼び出し
    batch_embeddings = embed_text(batch_texts)
    
    # 安全性チェック: API 呼び出しが失敗して空のリストが返された場合、
    # 最終的なリストの長さが DataFrame の長さと一致するように None でパディング
    if not batch_embeddings:
        print(f"Batch {i} to {i + BATCH_SIZE} failed. Padding with None.")
        batch_embeddings = [None] * len(batch_texts)
        
    all_embeddings.extend(batch_embeddings)
    
    # 1 分あたりのリクエスト数（RPM）の制限に達しないように少し待機
    # 特定の API クォータに基づいてこの値を調整
    time.sleep(1.0) 

# 4. 得られた埋め込みを DataFrame に割り当て
if len(all_embeddings) == len(amazon):
    amazon['amazon_embedding'] = all_embeddings
    print("Successfully added embeddings to the DataFrame!")
else:
    print(f"Length mismatch: {len(batch_embeddings)} embeddings for {len(amazon)} rows.")

# 列の幅を一時的に 100 文字に制限
with pd.option_context('display.max_colwidth', 100):
    display(amazon.head())

## 埋め込み類似度指標の実装
### ⚙️ 内積（Dot Product）の実装
2 つの $n$ 次元ベクトル $A$ と $B$ の **内積**（スカラー積とも呼ばれます）は、各ベクトルの対応する要素同士を掛け合わせ、それらの積を合計することによって計算されます。

$$A \cdot B = \sum_{i=1}^{n} a_i b_i = a_1 b_1 + a_2 b_2 + \dots + a_n b_n$$

ここで:
* $a_i$ と $b_i$ は、インデックス $i$ におけるベクトル $A$ と $B$ の個々のコンポーネント（要素）を表します。
* $n$ は次元の総数（埋め込みの長さ）です。

In [ ]:
#TODO: 内積を計算するコードを実装:
def calculate_dot_product(vec1, vec2):
    """
    Calculates the Dot Product between two embedding vectors.
    """
    ...

### 内積の実装のテスト

In [ ]:
# テスト 1: 直交
assert calculate_dot_product([1, 0], [0, 1]) == 0, "Failed Orthogonal Test"
# テスト 2: 標準
assert calculate_dot_product([1, 2, 3], [4, 5, 6]) == 32, "Failed Standard Test"
# テスト 3: 負の値
assert calculate_dot_product([2, 4, -1], [3, -2, 5]) == -7, "Failed Negative Test"
print("✅ All quick tests passed successfully!")

### ⚙️ コサイン類似度の実装

**コサイン類似度** は、ベクトルの大きさに関係なく、2 つのベクトルがどれだけ類似しているかを測定するために使用される指標です。多次元空間に投影された 2 つのベクトル間の角度のコサインを測定します。

* 📌 **主な用途:** 自然言語処理（NLP）やレコメンデーション システムで、ドキュメントやテキストの埋め込みを比較するために広く使用されます。
* ⚠️ **値の範囲:** **-1 から 1** の間の値を出力します。**1** はベクトルの方向が同一であることを意味し、**0** は直交（無相関）であることを意味し、**-1** は正反対であることを意味します。

---

#### 数式

2 つのベクトル $\mathbf{A}$ と $\mathbf{B}$ の間のコサイン類似度は次のように計算されます。

$$\text{Cosine Similarity} = \cos(\theta) = \frac{\mathbf{A} \cdot \mathbf{B}}{\|\mathbf{A}\| \|\mathbf{B}\|}$$

ここで:
* $\mathbf{A} \cdot \mathbf{B}$ はベクトルの **内積** です。
* $\|\mathbf{A}\|$ および $\|\mathbf{B}\|$ はベクトルの **ユークリッド ノルム（大きさ）** です。

---

#### コードの内訳

以下は、`calculate_cosine_similarity` 関数の各コンポーネントが数式にどのように対応しているかの概要です。

| ステップ / 行 | コード セグメント | 数学的な同等物 | 目的 / 説明 |
| :--- | :--- | :--- | :--- |
| **1. 内積** | `np.dot(embedding1, embedding2)` | $\mathbf{A} \cdot \mathbf{B}$ | 2 つの数値シーケンスの対応する要素の積の合計を計算します。 |
| **2. 大きさ** | `norm(embedding1) * norm(embedding2)` | $\|\mathbf{A}\| \times \|\mathbf{B}\|$ | 両方のベクトルの L2 ノルム（ユークリッド長）の積を計算します。 |
| **3. 安全性チェック** | `if magnitude == 0: return 0.0` | $N/A$ | 入力埋め込みの 1 つが全ゼロ ベクトルの場合に **ゼロ除算エラーを回避** します。 |
| **4. 最終除算** | `return dot_product / magnitude` | $\frac{\mathbf{A} \cdot \mathbf{B}}{\|\mathbf{A}\| \|\mathbf{B}\|}$ | 内積を大きさの積で割って、最終的なコサイン類似度スコアを取得します。 |

---

#### コードの実装

In [ ]:
#TODO: コサイン類似度を計算するコードを実装:
from numpy.linalg import norm
def calculate_cosine_similarity(embedding1, embedding2):
    ...

### コサイン類似度の実装のテスト

In [ ]:
# 1. 同一
assert abs(calculate_cosine_similarity([3, 4], [3, 4]) - 1.0) < 1e-9

# 2. 逆向き
assert abs(calculate_cosine_similarity([1, -1], [-1, 1]) - (-1.0)) < 1e-9

# 3. 直交
assert abs(calculate_cosine_similarity([1, 0], [0, 1]) - 0.0) < 1e-9

# 4. ゼロの安全性
assert calculate_cosine_similarity([0, 0], [1, 2]) == 0.0

print("✅ All quick checks passed!")

### ⚙️ ユークリッド距離の実装
**ユークリッド距離** は、ユークリッド空間内の 2 点間の直線距離を計算するために使用される最も一般的な指標です。

機械学習や自然言語処理では、2 つのベクトル埋め込み間の非類似度を測定するために使用されます。距離が `0` の場合は埋め込みが同一であることを意味し、値が大きいほど非類似度が大きいことを示します。

---

#### 数式

2 つの $n$ 次元ベクトル $A$ と $B$ に対するユークリッド距離は、数学的に次のように定義されます。

$$d(A, B) = \sqrt{\sum_{i=1}^{n} (a_i - b_i)^2}$$

#### ステップごとの内訳:
1. **要素ごとの減算 ($a_i - b_i$):** 2 つのベクトルの対応する次元間の差を求めます。
2. **差の 2 乗 ($(a_i - b_i)^2$):** 得られた各差を 2 乗します。これにより負の符号が排除され、より大きな差に対してより大きなペナルティが課されます。
3. **総和 ($\sum$):** すべての 2 乗された差を合計します。
4. **平方根 ($\sqrt{x}$):** 合計の平方根を取り、距離のスケールを元の単位次元に戻します。

---

#### ユークリッド距離の実装:

In [ ]:
#TODO: ユークリッド距離を計算するコードを実装:
def calculate_euclidean_distance(embedding1, embedding2):
    ...

### ユークリッド距離の実装のテスト

In [ ]:
# 1. 同一性
assert calculate_euclidean_distance(np.asarray([1, 2, 3]), np.asarray([1, 2, 3])) == 0.0

# 2. 既知の 3D 距離 (sqrt(1^2 + 1^2 + 1^2) = sqrt(3) ≈ 1.73205)
expected = 1.7320508075688772
assert abs(calculate_euclidean_distance(np.asarray([1, 1, 1]), np.asarray([2, 2, 2])) - expected) < 1e-9

# 3. 対称性
assert calculate_euclidean_distance(np.asarray([1, 5]), np.asarray([2, 10])) == calculate_euclidean_distance(np.asarray([2, 10]), np.asarray([1, 5]))

print("✅ All quick checks passed successfully!")

### 使用例
1. 検索クエリの埋め込みを生成する（最初の関数を使用）
2. 列全体に類似度関数を適用する
3. DataFrame をソートして、最も関連性の高い結果を上位に表示する

In [ ]:
amazon['cosine_similarity'] = amazon['amazon_embedding'].apply(
    lambda row_embedding: calculate_cosine_similarity(input_embd_vector, row_embedding)
)

amazon['dot_product'] = amazon['amazon_embedding'].apply(
    lambda row_embedding: calculate_dot_product(input_embd_vector, row_embedding)
)

amazon['euclidean_distance'] = amazon['amazon_embedding'].apply(
    lambda row_embedding: calculate_euclidean_distance(input_embd_vector, row_embedding)
)

amazon = amazon.sort_values(by='cosine_similarity', ascending=False)
# 列の幅を一時的に 100 文字に制限
with pd.option_context('display.max_colwidth', 100):
    display(amazon.head())

### 類似度分布の可視化
類似度分布のヒストグラムをプロットします

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# クリーンなビジュアル スタイルを設定
sns.set_theme(style="whitegrid")

# 図を初期化
plt.figure(figsize=(10, 6))

sns.histplot(
    amazon['cosine_similarity'],
    bins=100,
    kde=False,
    color='royalblue'
)

# 読みやすさのためにタイトルとラベルを追加
plt.title('Distribution of Cosine Similarity Scores', fontsize=14, pad=15)
plt.xlabel('Cosine Similarity (Higher = More Similar)', fontsize=12)
plt.ylabel('Number of Products', fontsize=12)

# プロットを表示
plt.show()

In [ ]:
# クリーンなビジュアル スタイルを設定

# 図を初期化
plt.figure(figsize=(10, 6))

sns.histplot(
    amazon['dot_product'],
    bins=100,
    kde=False,
    color='royalblue'
)

# 読みやすさのためにタイトルとラベルを追加
plt.title('Distribution of Dot Product', fontsize=14, pad=15)
plt.xlabel('Dot Product (Higher = More Similar)', fontsize=12)
plt.ylabel('Number of Products', fontsize=12)

# プロットを表示
plt.show()

In [ ]:
# クリーンなビジュアル スタイルを設定

# 図を初期化
plt.figure(figsize=(10, 6))

sns.histplot(
    amazon['euclidean_distance'],
    bins=100,
    kde=False,
    color='royalblue'
)

# 読みやすさのためにタイトルとラベルを追加
plt.title('Distribution of Euclidean Distance', fontsize=14, pad=15)
plt.xlabel('Euclidean Distance (Lower = More Similar)', fontsize=12)
plt.ylabel('Number of Products', fontsize=12)

# プロットを表示
plt.show()

### cosine_similarity 列の削除

In [ ]:
amazon = amazon.drop(columns=["cosine_similarity"])

### 切り捨てられたベクトル（MRL）の追加

In [ ]:
import time

start_time = time.perf_counter()

# 最初の 128 要素に切り捨て
amazon["amazon_embedding_mrl_128"] = amazon["amazon_embedding"].apply(
    lambda x: x[:128]
)

# 各ベクトルに L2 正規化を適用
amazon["amazon_embedding_l2_norm_mrl_128"] = amazon[
    "amazon_embedding_mrl_128"
].apply(l2_normalize)

end_time = time.perf_counter()
print(f"Elapsed time: {end_time - start_time:.4f} seconds")

## 製品ペアを見つけるためのセマンティック マッチングの実装
### 2 つ目のデータセットの読み込み
2 番目のテーブルには、Google のウェブサイトからスクレイピングされた製品情報に関して同様の内容が含まれています。

In [ ]:
google = pd.read_csv("../data/GoogleProducts.csv.gz")
google.columns = [
    "idGoogleBase",
    "google_title",
    "google_description",
    "google_manufacturer",
    "google_price",
]
google = google.dropna(subset=['google_description'])
google.head()

In [ ]:
import time
from tqdm import tqdm

BATCH_SIZE = 100 

# 1. タイトルと説明を結合（欠損値を適切に処理）
combined_text = google['google_title'].fillna('') + " " + google['google_description'].fillna('')

# リストに変換し、先ほど使用した切り捨てロジックを適用
MAX_CHARS = 4000
descriptions = [str(text)[:MAX_CHARS] for text in combined_text.tolist()]

all_embeddings = []

print(f"Starting to embed {len(descriptions)} documents in batches of {BATCH_SIZE}...")

# 3. チャンク単位でリストをループ処理
for i in tqdm(range(0, len(descriptions), BATCH_SIZE), desc="Embedding Batches"):
    # リストをスライスして現在のバッチを取得
    batch_texts = descriptions[i : i + BATCH_SIZE]
    
    # embed_text 関数を呼び出し
    batch_embeddings = embed_text(batch_texts)
    
    # 安全性チェック: API 呼び出しが失敗して空のリストが返された場合、
    # 最終的なリストの長さが DataFrame の長さと一致するように None でパディング
    if not batch_embeddings:
        print(f"Batch {i} to {i + BATCH_SIZE} failed. Padding with None.")
        batch_embeddings = [None] * len(batch_texts)
        
    all_embeddings.extend(batch_embeddings)
    
    # 1 分あたりのリクエスト数（RPM）の制限に達しないように少し待機
    # 特定の API クォータに基づいてこの値を調整
    time.sleep(1.0) 

# 4. 得られた埋め込みを DataFrame に割り当て
if len(all_embeddings) == len(google):
    google['google_embedding'] = all_embeddings
    print("Successfully added embeddings to the DataFrame!")
else:
    print(f"Length mismatch: {len(batch_embeddings)} embeddings for {len(google)} rows.")

# 列の幅を一時的に 100 文字に制限
with pd.option_context('display.max_colwidth', 100):
    display(google.head())

In [ ]:
import time

start_time = time.perf_counter()

# 最初の 128 要素に切り捨て
google["google_embedding_mrl_128"] = google["google_embedding"].apply(
    lambda x: x[:128]
)

# 各ベクトルに L2 正規化を適用
google["google_embedding_l2_norm_mrl_128"] = google[
    "google_embedding_mrl_128"
].apply(l2_normalize)

end_time = time.perf_counter()
print(f"Elapsed time: {end_time - start_time:.4f} seconds")

### グラウンドトゥルース（正解）データセットの読み込み

最後のテーブルには、同じ製品に対応する両ウェブサイトの製品情報のマッチングが含まれています（2 つのウェブサイトで表現が異なっている可能性があります）。

このデータセットを使用して、セマンティック マッチング アプローチを評価します。
この関数は、`matching` DataFrame 内の一致する製品 ID のサンプル（サイズは `SAMPLE_SIZE` で指定）を取得し、`google` および `amazon` DataFrame に含まれる Google と Amazon の製品情報を結合します。次に、一致する Google と Amazon の説明のペアを抽出し、`description_1`（Google）、`description_2`（Amazon）、およびターゲット列 `match`（現時点では一致するペアのみのため値は `yes`）を含む一致説明テーブルを作成します。
一致しない製品の説明ペアを作成するために、最初の説明を固定したまま 2 番目の説明列を並べ替え（置換）、この一致しない説明の新しい DataFrame を一致する説明の DataFrame に連結します。結果のテーブルをシャッフルし、2 つの同じサイズの DataFrame に分割して、評価セットおよびテストセットの分割としてディスクに保存します。

製品の説明として `title` 列のみを使用していることに注目してください。したがって、生データにはもっと多くの情報が存在します。それにもかかわらず、Gemini がテストセットで 100% に近いパフォーマンスを達成することを確認できます。これは驚くべきことです！

**注:** データの別のサンプルで評価セットとテストセットを再生成したい場合は、最後の行のコメントを解除してください。

In [ ]:
matching = pd.read_csv("../data/Amzon_GoogleProducts_perfectMapping.csv.gz")

# 一致する ID の DataFrame に製品情報を結合
matched_products = (
    matching.merge(right=amazon, how="inner", on="idAmazon")
    .merge(right=google, how="inner", on="idGoogleBase")
    .dropna()
)

matched_products.head()

## 📊 マッチング結果の評価とランキング

**`evaluate_matching_ranked`** は、反復的なペアワイズ比較アプローチを使用してエンティティ マッチングのパフォーマンス（例: Amazon の製品リスティングと Google の製品リスティングのペアリング）を評価し、**Top-K 正解率（Accuracy）** 指標を計算します。

---

### 主要なアルゴリズムの概念

#### 1. ペアワイズ比較行列（$O(N \times M)$ の計算量）
$N$ 個の Amazon アイテムと $M$ 個の Google アイテムを含むデータセットに対して、この関数はサイズ $N \times M$ の 2D グリッド（行列）を構築します。Amazon データセット内のすべてのアイテムは、ユーザー定義の `similarity_func` を使用して、Google データセット内のすべてのアイテムと体系的に比較されます。

$$\mathbf{S}_{i, j} = \text{similarity\_func}(\text{Amazon}_i, \text{Google}_j)$$

#### 2. ソート戦略
ソートの動作は、指標が **距離**（値が小さいほど良い）か **類似度**（値が大きいほど良い）かに応じて動的に適応します。

* **距離ベース ($is\_distance=True$):** インデックスを **昇順** でランク付けします。
* **類似度ベース ($is\_distance=False$):** インデックスを **降順** でランク付けします（行列を反転した負の行列 `-similarity_matrix` を使用して実現）。

---

### パラメータ設定

| パラメータ | 型 | デフォルト | 説明 |
| :--- | :--- | :--- | :--- |
| **`df`** | *pandas.DataFrame* | *必須* | 並列データ ストリーム（例: 正解ペアと識別子）を含む入力 DataFrame。 |
| **`col_amazon`** | *str* | *必須* | Amazon のテキスト / 埋め込みを含む DataFrame の列名。 |
| **`col_google`** | *str* | *必須* | Google のテキスト / 埋め込みを含む DataFrame の列名。 |
| **`similarity_func`** | *function* | *必須* | アイテムの比較に使用されるスコアリング関数（コサイン類似度、レーベンシュタイン距離など）。 |
| **`is_distance`** | *bool* | `False` | ソートの方向を決定します。距離指標（ユークリッド距離など）の場合は `True`、類似度指標（内積など）の場合は `False` に設定します。 |

---

### コード実行の内訳

アルゴリズムは、マッチング問題を次の 5 つの明確な操作ステップで処理します。

| ステップ | 操作 | 関連する主要変数 | 技術的な目的 |
| :--- | :--- | :--- | :--- |
| **1** | **初期化とセットアップ** | `amazon_data`, `google_data`, `google_ids` | 入力 DataFrame をコピーし、イテレーション中のメモリ アクセスを高速化するためにターゲットを軽量な Python リストに抽出します。 |
| **2** | **完全イテレーション行列** | `similarity_matrix` | ネストされたループを実行して、`similarity_func` を正確に $N \times M$ 回実行します。`time.perf_counter()` を使用してタイムスタンプを記録します。 |
| **3** | **インデックスのランキング** | `ranked_indices` | `axis=1` に沿って `np.argsort` を使用し、一致候補をランク付けします。距離の場合は値が小さいものが上位になり、類似度の場合は値が大きいものが上位になります。 |
| **4** | **行の評価** | `all_ranked_ids`, `correct_match_ranks` | 各 Amazon の行を反復処理して、正解のターゲット Google ID がどこにランク付けされているかを特定し、生の配列インデックスを **1 から始まるランク（順位）** に変換します。 |
| **5** | **指標と出力の生成** | `top_1_acc` から `top_20_acc` | Top-K 正解率（正解の一致が上位 $K$ 個の選択肢に含まれた割合）を計算し、統計情報をコンソールに出力します。 |

---

### 出力指標の解釈

* **最良一致の正解率（Top-1 Accuracy）:** モデルの第 1 推奨（1 位）が真の正解ターゲットと一致した割合。
* **Top-K 正解率（Top-K Accuracy, $K \in \{3, 5, 10, 20\}$）:** 正解の一致が上位 $K$ 個にランク付けされた予測内に含まれていた割合:
  
$$\text{Top-}K\text{ Accuracy} = \frac{1}{N} \sum_{i=1}^{N} \mathbb{I}(\text{Rank}_i \le K) \times 100$$

*（ここで $\mathbb{I}$ は、条件が真の場合は 1 を、偽の場合は 0 を返す指示関数です）。*

In [ ]:
import time
import numpy as np
import pandas as pd

def evaluate_matching_ranked(df, col_amazon, col_google, similarity_func, is_distance=False):
    """
    Evaluates matches between Amazon and Google columns and ranks all combinations.
    
    Returns:
    - result_df: DataFrame with appended ranking lists (1 to N) and Top-K metrics.
    """
    result_df = df.copy()
    
    amazon_data = result_df[col_amazon].tolist()
    google_data = result_df[col_google].tolist()
    
    # Extract the ground truth IDs to map back to our predictions
    google_ids = result_df['idGoogleBase'].values 
    
    N = len(amazon_data)
    M = len(google_data)
    
    metric_type = "distance" if is_distance else "similarity"
    print(f"Calculating {metric_type} iteratively for {N} x {M} ({N * M:,}) combinations...")
    
    similarity_matrix = np.zeros((N, M))
    start_time = time.perf_counter()

    # --- Full Iteration Logic ---
    # We compare every Amazon item to every Google item.
    for i in range(N):
        for j in range(M): 
            similarity_matrix[i, j] = similarity_func(amazon_data[i], google_data[j])
            
    end_time = time.perf_counter()

    # --- Calculate Performance Metrics ---
    execution_time = end_time - start_time
    
    # --- Ranking Logic ---
    # np.argsort returns the indices that would sort the array. 
    if is_distance:
        # For distances, lower is better (ascending sort)
        ranked_indices = np.argsort(similarity_matrix, axis=1)
    else:
        # For similarities, higher is better (descending sort)
        # We use negative similarity_matrix to sort highest-to-lowest
        ranked_indices = np.argsort(-similarity_matrix, axis=1)

    # Pre-allocate lists for our new DataFrame columns
    all_ranked_ids = []
    all_ranked_scores = []
    correct_match_ranks = []

    for i in range(N):
        # Extract the sorted indices for this specific Amazon row
        row_sorted_indices = ranked_indices[i]
        
        # Reorder the IDs and Scores based on the sorted indices
        sorted_scores = similarity_matrix[i, row_sorted_indices]
        sorted_ids = google_ids[row_sorted_indices]
        
        all_ranked_ids.append(sorted_ids.tolist())
        all_ranked_scores.append(sorted_scores.tolist())
        
        # Calculate what rank the actual correct ID achieved (1-based indexing)
        true_id = result_df['idGoogleBase'].iloc[i]
        rank_pos = np.where(sorted_ids == true_id)[0]
        
        if len(rank_pos) > 0:
            correct_match_ranks.append(rank_pos[0] + 1) # +1 so rank starts at 1, not 0
        else:
            correct_match_ranks.append(None)
            
    # --- Assigning to DataFrame ---
    result_df['ranked_predicted_idGoogleBase'] = all_ranked_ids
    result_df['ranked_scores'] = all_ranked_scores
    result_df['correct_match_rank'] = correct_match_ranks
    
    # Preserve original logic: isolate the #1 best match
    result_df['predicted_idGoogleBase'] = [ids[0] for ids in all_ranked_ids]
    result_df['best_match_score'] = [scores[0] for scores in all_ranked_scores]
    result_df['is_correct_match'] = result_df['idGoogleBase'] == result_df['predicted_idGoogleBase']
    
    # Only calculate the diagonal pair score if the matrix is perfectly square
    if N == M:
        result_df['pair_score'] = similarity_matrix.diagonal()
    else:
        result_df['pair_score'] = None
    
    total_comparisons = N * M
    
    # Calculate Top-K Accuracies
    top_1_acc = (result_df['correct_match_rank'] == 1).mean() * 100
    top_3_acc = (result_df['correct_match_rank'] <= 3).mean() * 100
    top_5_acc = (result_df['correct_match_rank'] <= 5).mean() * 100
    top_10_acc = (result_df['correct_match_rank'] <= 10).mean() * 100
    top_20_acc = (result_df['correct_match_rank'] <= 20).mean() * 100
    
    print("-" * 40)
    print(f"Execution time: {execution_time:.4f} seconds")
    print(f"Total pairwise comparisons calculated: {total_comparisons:,}")
    print(f"Average time per comparison: {(execution_time / total_comparisons) * 1e6:.2f} microseconds")
    print("-" * 40)
    print("Model Accuracy (Where did the correct match rank?):")
    print(f"Best Match Accuracy: {top_1_acc:.2f}%")
    print(f"Top-3 Accuracy: {top_3_acc:.2f}%")
    print(f"Top-5 Accuracy: {top_5_acc:.2f}%")
    print(f"Top-10 Accuracy: {top_10_acc:.2f}%")
    print(f"Top-20 Accuracy: {top_20_acc:.2f}%")
    print("-" * 40)
    
    return result_df

### 実験: セマンティック マッチングの構成

コードは、ペアごとに評価パイプライン `evaluate_matching_ranked` を呼び出すことによって、**埋め込み** と **距離 / 類似度指標** のすべての組み合わせを体系的に評価します。

In [ ]:
import time

# 1. Define the 3 Embedding Configurations to test
embedding_configs = [
    "embedding",  # Full Dimensionality (3072)
    "embedding_mrl_128",  # Matryoshka Truncated (128) - Raw
    "embedding_l2_norm_mrl_128",  # Matryoshka Truncated (128) - L2 Normalized
]

# 2. Define the 3 Similarity Functions to test
similarity_functions = [
    ("Cosine Similarity", calculate_cosine_similarity, False),
    ("Dot Product", calculate_dot_product, False),
    ("Euclidean Distance", calculate_euclidean_distance, True),
]

# Structure to store all aggregated results
benchmark_results = []

# Nested Loop: 3 Embedding Spaces x 3 Distance Metrics = 9 Evaluations
for embd_config in embedding_configs:
    for sim_name, sim_func, is_distance in similarity_functions:

        config_label = f"{embd_config} + {sim_name}"
        print(f"\n[RUNNING] Configuration: {config_label}")

        # 呼び出し全体の実行時間を計測
        start_time = time.perf_counter()

        # 評価メソッドを実行
        evaluated_df = evaluate_matching_ranked(
            df=matched_products,
            col_amazon=f"amazon_{embd_config}",
            col_google=f"google_{embd_config}",
            similarity_func=sim_func,
            is_distance=is_distance,
        )

        total_time = time.perf_counter() - start_time

        # 返された DataFrame からパフォーマンス指標を直接取得
        top_1_acc = (evaluated_df["correct_match_rank"] == 1).mean() * 100
        top_3_acc = (evaluated_df["correct_match_rank"] <= 3).mean() * 100
        top_5_acc = (evaluated_df["correct_match_rank"] <= 5).mean() * 100
        top_10_acc = (evaluated_df["correct_match_rank"] <= 10).mean() * 100
        top_20_acc = (evaluated_df["correct_match_rank"] <= 20).mean() * 100

        # すべてのメタデータと指標を追加
        benchmark_results.append(
            {
                "Embedding Config": embd_config,
                "Similarity Metric": sim_name,
                "Configuration": config_label,
                "Top-1 Accuracy (%)": top_1_acc,
                "Top-3 Accuracy (%)": top_3_acc,
                "Top-5 Accuracy (%)": top_5_acc,
                "Top-10 Accuracy (%)": top_10_acc,
                "Top-20 Accuracy (%)": top_20_acc,
                "Execution Time (s)": total_time,
            }
        )

# 最終的な比較 DataFrame を作成
comparison_df = pd.DataFrame(benchmark_results)
print("\n[SUCCESS] All configurations evaluated successfully!")

### 各埋め込み / 類似度構成について Top-1 から Top-20 までの正解率の推移を可視化

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_all_configs_grid(df):
    """
    単一の埋め込み / 類似度構成について、Top-1 から Top-20 までの
    正解率の推移を示す 3x3 グリッドの個別グラフをプロットします。
    """
    sns.set_theme(style="whitegrid")
    
    # 一意の構成を特定（3 つの埋め込み構成 * 3 つの類似度構成 = 合計 9 構成）
    configs = df["Configuration"].unique()
    num_configs = len(configs)
    
    # グリッドの寸法を動的に計算
    cols = 3
    rows = (num_configs + cols - 1) // cols
    
    # クリーンで広々とした図のレイアウトを初期化
    fig, axes = plt.subplots(rows, cols, figsize=(20, 5 * rows), sharey=True)
    axes = axes.flatten()  # イテレーションを容易にするためにグリッドを 1D 配列にフラット化
    
    # 指標とそれに対応する DataFrame 内の列を定義
    metrics = ["Top-1", "Top-3", "Top-5", "Top-10", "Top-20"]
    metric_cols = [f"{m} Accuracy (%)" for m in metrics]
    
    for i, config in enumerate(configs):
        ax = axes[i]
        
        # この特定の構成に属する行を抽出
        config_row = df[df["Configuration"] == config]
        accuracy_values = config_row[metric_cols].values[0]
        
        # シーケンシャルな棒グラフをプロット
        sns.barplot(x=metrics, y=accuracy_values, ax=ax, hue=metrics, palette="crest", legend=False)
        
        # サブプロットのカスタマイズ
        ax.set_title(config, fontsize=12, fontweight='bold', pad=10)
        ax.set_ylim(0, 115)
        ax.set_ylabel("Accuracy (%)" if i % cols == 0 else "")  # Y 軸ラベルは左列のみに表示
        
        ax.set_xlabel("Accuracy Threshold", fontsize=9, fontweight='bold')
        
        # 各バーの真上に値ラベルを描画
        for bar in ax.patches:
            height = bar.get_height()
            ax.text(
                bar.get_x() + bar.get_width() / 2.0, 
                height + 2, 
                f'{height:.1f}%', 
                ha='center', 
                va='bottom', 
                fontsize=9, 
                fontweight='bold'
            )
            
    # グリッドサイズが不均一な場合、余分な空のサブプロットを削除
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])
        
    plt.suptitle("Top-K Accuracy Progression per Configuration", fontsize=20, fontweight='bold', y=0.99)
    plt.tight_layout()
    plt.show()

# グリッド可視化ツールを実行
plot_all_configs_grid(comparison_df)

### 構成の組み合わせ全体で正解率と実行時間を比較

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_configuration_dashboard_vertical(df):
    """
    構成の組み合わせ全体で正解率と実行時間を比較するための、
    垂直に積み重ねられたサブプロットを持つデュアルプロット ダッシュボードを生成します。
    """
    # クリーンでモダンな美観を適用
    sns.set_theme(style="whitegrid")
    
    # 2 行 1 列に再構成
    fig, axes = plt.subplots(2, 1, figsize=(12, 16))
    
    # --- プロット 1: 複数指標の正解率比較（上部プロット） ---
    accuracy_cols = ["Top-1 Accuracy (%)", "Top-5 Accuracy (%)", "Top-20 Accuracy (%)"]
    df_melted = df.melt(
        id_vars=["Configuration"],
        value_vars=accuracy_cols,
        var_name="Accuracy Metric",
        value_name="Accuracy (%)"
    )
    
    sns.barplot(
        ax=axes[0],
        data=df_melted,
        x="Configuration",
        y="Accuracy (%)",
        hue="Accuracy Metric",
        palette="Blues_d"
    )
    
    axes[0].set_title("Accuracy Metrics by Configuration", fontsize=16, fontweight='bold', pad=15)
    axes[0].set_xlabel("")
    axes[0].set_ylabel("Accuracy (%)", fontsize=12, fontweight='bold')
    axes[0].set_ylim(0, 115)
    axes[0].tick_params(axis='x', rotation=30, labelsize=10)
    
    # バーの上に正解率のパーセンテージを注記
    for container in axes[0].containers:
        axes[0].bar_label(container, fmt='%.1f%%', padding=3, fontsize=9)

    # --- プロット 2: 計算速度（下部プロット） ---
    sns.barplot(
        ax=axes[1],
        data=df,
        x="Configuration",
        y="Execution Time (s)",
        hue="Configuration",
        palette="Reds_d",
        legend=False
    )
    
    axes[1].set_title("Execution Time Comparison (Lower is Better)", fontsize=16, fontweight='bold', pad=15)
    axes[1].set_xlabel("Configuration Combo", fontsize=12, fontweight='bold', labelpad=15)
    axes[1].set_ylabel("Execution Time (Seconds)", fontsize=12, fontweight='bold')
    axes[1].tick_params(axis='x', rotation=30, labelsize=10)
    
    # バーの上に秒数を注記
    for container in axes[1].containers:
        axes[1].bar_label(container, fmt='%.3fs', padding=3, fontsize=10)

    # レイアウトを整えて表示
    plt.tight_layout()
    plt.show()

# 垂直可視化ダッシュボードを実行
plot_configuration_dashboard_vertical(comparison_df)

Copyright 2026 Google LLC

Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at

https://www.apache.org/licenses/LICENSE-2.0
Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.